# 05 — Execution, Evaluation & Integration
**Agentic X | Studio 67** — Real components only. Run from the repository root.

## 1. Objective
Run actual Studio 67 scenarios, evaluate source-grounded results, record LangSmith traces if configured, and export data for UI consumption. Do not confuse reference QA labels with measured model performance.

In [ ]:
from pathlib import Path
import os,sys,time,json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'core').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from core.environment import load_dataset
from core.runner import build_engine, run_case
from langsmith import traceable
print('LangSmith enabled:',os.getenv('LANGSMITH_TRACING','false'))

## 2. Three required demonstration scenarios
Select one real case from each required category. The 150-case dataset provides synthetic test inputs, not measured performance.

In [ ]:
cases,options=load_dataset()
DEMO_CASES=['SCR-001','BDG-001','SFT-001']
assert all(case_id in set(cases['case_id']) for case_id in DEMO_CASES)
print(cases[cases['case_id'].isin(DEMO_CASES)][['case_id','scenario','incident']].to_string(index=False))

## 3. Build real engine and trace executions
Requires actual LLM access. LangSmith captures traces if `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true` and `LANGSMITH_PROJECT` are set.

In [ ]:
engine=build_engine()
@traceable(name='studio67_case_execution',run_type='chain')
def traced_run(case_id):
    result,ms=run_case(case_id,engine)
    return {'result':result.model_dump(),'duration_ms':ms}
scenario_runs={case_id:traced_run(case_id) for case_id in DEMO_CASES}
for cid,item in scenario_runs.items():
    print(cid,item['result']['status'],item['result']['selected_option_id'],item['duration_ms'])

## 4. Verify scenario traces
At least one demo scenario must present an agent conflict or failure. Only claim conflict when its real agent refusals/messages demonstrate it.

In [ ]:
for cid,item in scenario_runs.items():
    r=item['result']
    refusals=[m for m in r['messages'] if m['performative']=='REFUSE']
    print(cid,'Refusals:',len(refusals),'ACL messages:',len(r['messages']))
    assert r['is_mock'] is False
    assert r['messages']
assert any(any(m['performative']=='REFUSE' for m in x['result']['messages']) for x in scenario_runs.values()), 'Demonstrate at least one real conflict/refusal in report'

## 5. Batch evaluation: 150 cases
This cell makes real LLM calls. It may consume API credits and take time. Start with a small subset before opting into all 150.

In [ ]:
RUN_ALL_150=False  # Set True deliberately after confirming provider budget and rate limits.
ids=cases['case_id'].tolist() if RUN_ALL_150 else cases['case_id'].head(5).tolist()
records=[]
for cid in ids:
    try:
        data=traced_run(cid)
        r=data['result']
        records.append({'case_id':cid,'status':r['status'],'selected_option_id':r['selected_option_id'],
                        'rounds':r['rounds'],'messages_count':len(r['messages']),'duration_ms':data['duration_ms'],'error':None})
    except Exception as exc:
        records.append({'case_id':cid,'status':'FAILED','selected_option_id':None,'rounds':None,
                        'messages_count':None,'duration_ms':None,'error':str(exc)})
evaluation_df=pd.DataFrame(records)
print('Tested:',len(evaluation_df),'of',len(cases))
print(evaluation_df['status'].value_counts())

## 6. Evaluation metrics
Report observed rates only for executed runs, with denominator and failure counts. The dataset reference feasibility is not a measured MAS success rate.

In [ ]:
n=len(evaluation_df)
assert n>0
print('Measured successful agreement rate:',round((evaluation_df['status']=='AGREED').sum()/n*100,2),'%')
print('Execution errors:',evaluation_df['error'].notna().sum())
print('Mean latency (ms):',evaluation_df['duration_ms'].mean())

## 7. Export genuine results for interface integration
The finished UI source is not included here. Export structured JSON so that the actual interface can consume the live message log and statuses.

In [ ]:
out=ROOT/'results';out.mkdir(exist_ok=True)
(out/'scenario_runs.json').write_text(json.dumps(scenario_runs,indent=2,ensure_ascii=False),encoding='utf-8')
evaluation_df.to_csv(out/'measured_results.csv',index=False)
print('Exports:',out/'scenario_runs.json',out/'measured_results.csv')

## 8. Final acceptance checks
Use the final Python application to show each agent's live status, actual ACL message log, run controls and adjustable variables. Collect screenshots after real execution.

In [ ]:
assert set(DEMO_CASES).issubset(scenario_runs)
assert all(r['result']['is_mock'] is False for r in scenario_runs.values())
print('Scenario exports are real engine results. UI integration and complete 150-case run require separate verification.')